<a href="https://colab.research.google.com/github/SwaksharDebnath/CRF-for-Bangla-Ancholik-NER-/blob/main/EDA_chittagong.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import os
import random

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

df = pd.read_csv('/content/drive/MyDrive/NER_Dataset/CSV_data/Chittagong_NER.csv')


df['Sentence #'] = df['Sentence #'].ffill()

sentences = [group for _, group in df.groupby('Sentence #', sort=False)]


random.seed(42)
random.shuffle(sentences)

# Split 80:20
split_idx = int(len(sentences) * 0.8)
train_sentences = sentences[:split_idx]
test_sentences = sentences[split_idx:]

train_df = pd.concat(train_sentences)
test_df = pd.concat(test_sentences)


def revert_format(data):
    mask = data['Sentence #'] != data['Sentence #'].shift(1)
    data_out = data.copy()
    data_out.loc[~mask, 'Sentence #'] = ''
    return data_out

train_df = revert_format(train_df)
test_df = revert_format(test_df)


train_df.to_csv('/content/drive/MyDrive/NER_Dataset/CSV_data/train/Chittagong_NER_train.csv', index=False)
test_df.to_csv('/content/drive/MyDrive/NER_Dataset/CSV_data/test/Chittagong_NER_test.csv', index=False)

print(f"Total sentences: {len(sentences)}")
print(f"Train sentences: {len(train_sentences)} (80%)")
print(f"Test sentences: {len(test_sentences)} (20%)")


Total sentences: 3481
Train sentences: 2784 (80%)
Test sentences: 697 (20%)


In [ ]:
import re
import pandas as pd # Ensure pandas is imported as it's used in this function

def convert_csv_to_conll(csv_path, output_path):
    df = pd.read_csv(csv_path)

    with open(output_path, 'w', encoding='utf-8') as f:
        first_word = True

        for index, row in df.iterrows():
            sentence_id = row['Sentence #']
            original_word = row['chittagong_word']
            original_tag = row['bio_tag']

            if pd.isna(original_word): # Skip if word is NaN
                continue

            # Clean word and tag to remove newlines, carriage returns, and excessive spaces
            # by replacing all whitespace sequences with a single space and stripping.
            word = re.sub(r'\s+', ' ', str(original_word)).strip()
            tag = re.sub(r'\s+', ' ', str(original_tag)).strip()

            # If word became empty string after cleaning (e.g., was just newlines or spaces)
            if not word:
                # If word is empty, but tag is not, it should still be written (e.g., ' _ _ O')
                # If both are empty, it's effectively a skipped line already by pd.isna(original_word) if it was NaN
                # If it was a non-NaN string that became empty after strip/replace, we should still output it with its tag.
                pass # Keep word as ''


            if pd.notna(sentence_id) and str(sentence_id).strip().startswith("Sentence:"):

                if not first_word:
                    f.write("\n")


            f.write(f"{word} _ _ {tag}\n")

            first_word = False

        # Add a final newline at the end of the file
        f.write("\n")

    print(f"Conversion complete! File saved to {output_path}")

In [ ]:
convert_csv_to_conll('/content/drive/MyDrive/NER_Dataset/CSV_data/train/Chittagong_NER_train.csv', '/content/drive/MyDrive/NER_Dataset/ConvertedTxtData/train/Chittagong_NER_train.txt')
convert_csv_to_conll('/content/drive/MyDrive/NER_Dataset/CSV_data/test/Chittagong_NER_test.csv', '/content/drive/MyDrive/NER_Dataset/ConvertedTxtData/test/Chittagong_NER_test.txt')

Conversion complete! File saved to /content/drive/MyDrive/NER_Dataset/ConvertedTxtData/train/Chittagong_NER_train.txt
Conversion complete! File saved to /content/drive/MyDrive/NER_Dataset/ConvertedTxtData/test/Chittagong_NER_test.txt


In [ ]:
def load_dataset(filename):
    with open(filename) as file:
        lines = [x.strip() for x in file.readlines()]

    words = []
    labels = []

    total = 0
    for l in lines:

        if l == "":
            total += 1
            continue
        else:
            words += [l.split(" _ _ ")[0]]
            labels += [l.split(" _ _ ")[1]]

    total+=1
    label_set = list(set(labels))
    label_types = [x.split('-')[1] for x in labels if len(x) > 1]

    print(filename)
    print('total:', total)
    print('label set:', label_set)
    print('label types:', set(label_types))
    print('label distribution:')
    for l in sorted(label_set):
        if l != 'X':
            print(l, ":",  "%.2f" % (labels.count(l)/ len(labels) * 100), "%")

    print('label types distribution:')
    for l in set(label_types):
        print(l, ":", "%.2f" % (label_types.count(l) / len(labels) * 100), "%")
    return words, labels, label_set

In [ ]:
train_words, train_labels, train_label_set = load_dataset('/content/drive/MyDrive/NER_Dataset/ConvertedTxtData/train/Chittagong_NER_train.txt')

IndexError: list index out of range

In [ ]:
test_words, test_labels, test_label_set = load_dataset('/content/drive/MyDrive/NER_Dataset/ConvertedTxtData/test/Chittagong_NER_test.txt')

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
import seaborn as sns


def plot_all_labels(label_set, all_labels):

    index_to_labels = {}
    for i in range(len(label_set)):
        index_to_labels[i] = label_set[i]

    print(index_to_labels)

    label_to_index = {}
    for key in index_to_labels:
        label_to_index[index_to_labels[key]] = key
    print(label_to_index)

    arr = np.zeros(len(label_set))
    label_counts = Counter(all_labels)
    for key, value in label_counts.items():
        arr[label_to_index[key]] = value


    plt.figure(figsize=(30,5))
    ax = sns.barplot(x=np.arange(len(label_set)),y=arr)
    ax.set_xticklabels(list(index_to_labels.values()), fontsize=15, rotation=40, ha="right")
    ax.set(xlabel='Classes', ylabel='Class Counts')
    plt.show()

In [ ]:
def plot_only_ne(label_set, all_labels):

    index_to_labels = {}
    i=0
    for x in label_set:
        if x == 'O':
            continue
        index_to_labels[i] = x
        i += 1

    print(index_to_labels)

    label_to_index = {}
    for key in index_to_labels:
        label_to_index[index_to_labels[key]] = key
    print(label_to_index)

    arr = np.zeros(len(label_set)-1)
    label_counts = Counter(all_labels)
    for key, value in label_counts.items():
        if key == 'O':
            continue
        arr[label_to_index[key]] = value


    plt.figure(figsize=(30,5))
    ax = sns.barplot(x=np.arange(len(label_set)-1),y=arr)
    ax.set_xticklabels(list(index_to_labels.values()), fontsize=15, rotation=40, ha="right")
    ax.set(xlabel='Classes', ylabel='Class Counts')
    plt.show()

In [ ]:
plot_all_labels(train_label_set, train_labels)

In [ ]:
plot_only_ne(train_label_set, train_labels)

In [ ]:
plot_all_labels(test_label_set, test_labels)

In [ ]:
plot_only_ne(test_label_set, test_labels)